In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

def clean_station_data(raw_list, month_names):
    parsed_values = []
    invalid_indices = []

    # First pass: convert valid strings to integers, flag invalid ones
    for idx, val in enumerate(raw_list):
        try:
            # .strip() handles whitespace like " 20 " before casting to int
            parsed_values.append(int(val.strip()))
        except ValueError:
            parsed_values.append(None)
            invalid_indices.append(idx)

    # Calculate rounded mean of valid entries
    valid_only = [v for v in parsed_values if v is not None]
    imputed_mean = round(sum(valid_only) / len(valid_only))

    # Second pass: impute missing values and document them
    imputed_months = []
    for idx in invalid_indices:
        parsed_values[idx] = imputed_mean
        imputed_months.append(month_names[idx])

    print(f"Imputed value used: {imputed_mean} mm")
    print(f"Months imputed for Machakos: {', '.join(imputed_months)}")
    return parsed_values

machakos_clean = clean_station_data(machakos_raw, months)

Imputed value used: 68 mm
Months imputed for Machakos: April, September


In [2]:
all_rows = clean_rows + [machakos_clean]
R = np.array(all_rows, dtype=int)

print("Array shape:", R.shape)
print("Array dtype:", R.dtype)
print("\nRainfall Matrix (mm):\n", R)

Array shape: (5, 12)
Array dtype: int64

Rainfall Matrix (mm):
 [[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


In [3]:
# Task 3: Annual total per county (sum across columns, axis=1) & monthly average (axis=0)
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("--- Annual Totals by County ---")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\n--- Average Monthly Rainfall Across All Counties ---")
for month, avg in zip(months, monthly_averages):
    print(f"{month}: {avg:.1f} mm")

# Task 4: Wettest month per county using np.argmax(axis=1)
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

print("\n--- Wettest Month per County ---")
for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")

# Task 5: Count dry months (< 50 mm) using a boolean mask
dry_mask = R < 50
dry_month_counts = dry_mask.sum(axis=1)

print("\n--- Dry Months (< 50 mm) per County ---")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} months")

# Task 6: Best 2 consecutive months (Planting Window) using shifted-array addition
two_month_sums = R[:, :-1] + R[:, 1:]
best_window_starts = np.argmax(two_month_sums, axis=1)
planting_windows = [f"{months[i]}-{months[i+1]}" for i in best_window_starts]

print("\n--- Recommended Planting Windows ---")
for county, window, peak_rain in zip(counties, planting_windows, two_month_sums.max(axis=1)):
    print(f"{county}: {window} ({peak_rain} mm combined)")

--- Annual Totals by County ---
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

--- Average Monthly Rainfall Across All Counties ---
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm

--- Wettest Month per County ---
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November

--- Dry Months (< 50 mm) per County ---
Nairobi: 4 months
Nakuru: 3 months
Kisumu: 0 months
Mombasa: 2 months
Machakos: 6 months

--- Recommended Planting Windows ---
Nairobi: April-May (395 mm combined)
Nakuru: April-May (265 mm combined)
Kisumu: April-May (420 mm combined)
Mombasa: April-May (390 mm combined)
Machakos: November-December (278 mm combined)


In [4]:
output_filename = "rainfall_summary.csv"
headers = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open(output_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    for row in zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows):
        writer.writerow(row)

# Verify the file is readable and properly formatted
with open(output_filename, mode="r", encoding="utf-8") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December



Planting Advisory for Farmer-Support NGO

Based on local weather station records, farmers in Nairobi, Nakuru, Kisumu, and Mombasa should prepare inputs ahead of the April–May long rains, which represent the peak two-month planting window across these four counties. Because Nakuru experiences year-round moisture with zero dry months below 50 mm, farmers there have higher flexibility for staggered planting compared to Nairobi and Mombasa, which experience four dry months annually. For Machakos, available records indicate the optimal two-month planting window occurs later in the year during November–December (278 mm combined), with five dry months requiring drought-tolerant crop varieties. Data-Quality Caveat: Manual entry errors at the Machakos weather station left April and September readings missing; these were imputed using the station's annual monthly average (68 mm), meaning Machakos's early-year rainfall may be underestimated if April experienced typical regional long rains.